# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth reviewing if it's genuinely visible (enough real search impressions to matter), ranks reasonably well already (page 1), but still underperforms CTR for its position tier meaning the ranking effort is there, but something (title, snippet, intent mismatch) is stopping people from clicking.

**Reason code:** `visible_page1_low_ctr`: the only reason code this rule can output, since it's one rule with one condition set.

**Action label:** `review_title_and_snippet`

**Why this rule, not staleness:**: the staleness signal tested MIXED/unreliable (see verdicts above), so this rule avoids it entirely and leans only on the CONFIRMED signal CTR vs. position.

In [5]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

!pip install -q duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready.")

DuckDB ready.


### Signal check 1: Staleness vs. CTR (flag-linked — behind FlyRank's refresh flags)

In [6]:
signal1 = con.execute("""
    SELECT
        CASE
            WHEN days_stale < 90 THEN 'fresh'
            WHEN days_stale < 180 THEN 'aging'
            ELSE 'stale'
        END as freshness_bucket,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_clicks*1.0/f.gsc_impressions ELSE NULL END) as avg_ctr,
        COUNT(*) as n
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN (
        SELECT content_hash_id, DATE_DIFF('day', content_updated_date, DATE '2026-03-31') as days_stale
        FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) c ON f.content_hash_id = c.content_hash_id
    GROUP BY freshness_bucket
    ORDER BY avg_ctr DESC
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  freshness_bucket   avg_ctr        n
0            aging  0.011478   107575
1            fresh  0.003059  9615509
2            stale  0.002125   118294


### Signal check 2: CTR vs. position (behind the CTR-fix logic)

In [7]:
signal2 = con.execute("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'top_3'
            WHEN gsc_avg_position <= 10 THEN 'page_1'
            ELSE 'beyond_page_1'
        END as position_bucket,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks*1.0/gsc_impressions ELSE NULL END) as avg_ctr,
        COUNT(*) as n
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_avg_position > 0
    GROUP BY position_bucket
    ORDER BY avg_ctr DESC
""").df()
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket   avg_ctr        n
0           top_3  0.004918   564173
1          page_1  0.003473  1456122
2   beyond_page_1  0.001828  1427577


### Verdicts

**Signal 1: Staleness vs. CTR: MIXED.** The pattern is non monotonic (aging=0.0115 > fresh=0.0031 > stale=0.0021) staleness does not cleanly predict CTR. This likely connects to a known data quality issue from w03: content_updated_date is unreliable for a large share of pages (batch-default dates), which would corrupt any staleness based grouping. Decision: staleness will NOT be used as a primary rule signal.

**Signal 2: CTR vs. Position: CONFIRMED** (flag-linked, behind the CTR-fix logic). Clean, monotonic decline: top_3=0.0049, page_1=0.0035, beyond_page_1=0.0018 (n=3.4M+ total). Matches the same CTR-cliff pattern proven in Notebook 01. This signal is trustworthy and will anchor the rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
rule_df = con.execute("""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_impressions) as impressions_total,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks)*1.0/SUM(gsc_impressions) ELSE NULL END as ctr
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

visible = (rule_df["impressions_total"] >= 500).astype(int)
good_position = (rule_df["avg_position"] <= 10).astype(int)
low_ctr = (rule_df["ctr"].fillna(0) < 0.003).astype(int)  # below the page_1 average CTR from signal 2

rule_df["score"] = visible * good_position * low_ctr * rule_df["impressions_total"]
rule_df["reason_code"] = "visible_page1_low_ctr"
rule_df["action"] = "review_title_and_snippet"

import os
os.makedirs("work/outputs", exist_ok=True)
ranked = rule_df.sort_values("score", ascending=False)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote", len(ranked), "rows.")
print("Rows with score > 0:", (ranked["score"] > 0).sum())
ranked.head(10)

Wrote 175304 rows.
Rows with score > 0: 23324


,content_hash_id,avg_position,impressions_total,ctr,score,reason_code,action
17850,content_44f34c0a90047651,7.346909,212404.0,0.000113,212404.0,visible_page1_low_ctr,review_title_and_snippet
61650,content_8d7d99f109e19aa2,2.563756,203497.0,0.001420,203497.0,visible_page1_low_ctr,review_title_and_snippet
22155,content_b99ea6861864dea5,4.450106,194337.0,0.001858,194337.0,visible_page1_low_ctr,review_title_and_snippet
66979,content_acbcc847f8996314,3.361195,170808.0,0.001534,170808.0,visible_page1_low_ctr,review_title_and_snippet
99100,content_471d9cabce329a66,4.656030,164885.0,0.002402,164885.0,visible_page1_low_ctr,review_title_and_snippet
517,content_fd2117c2c6790e4b,3.391428,151166.0,0.002699,151166.0,visible_page1_low_ctr,review_title_and_snippet
37894,content_34a70fea29d15f24,3.219473,143019.0,0.000301,143019.0,visible_page1_low_ctr,review_title_and_snippet
25153,content_e241d6415ac9e534,3.276016,142304.0,0.002410,142304.0,visible_page1_low_ctr,review_title_and_snippet
82021,content_f43118e089ecc69a,5.036458,139417.0,0.001370,139417.0,visible_page1_low_ctr,review_title_and_snippet
21980,content_f352b7cfd0b2f434,3.268127,136098.0,0.002109,136098.0,visible_page1_low_ctr,review_title_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 review

1. **content_44f34c0a90047651**: position 7.3, 212K impressions, CTR 0.011%. Action: review_title_and_snippet. **Wrong if:** this page's low CTR is due to a rich SERP feature (featured snippet, People Also Ask) stealing clicks above it, not a bad title.
2. **content_8d7d99f109e19aa2**: position 2.6, 203K impressions, CTR 0.14%. **Wrong if:** the query intent is informational-only (no click needed, answer shown in snippet): a title rewrite wouldn't fix that.
3. **content_b99ea6861864dea5**: position 4.5, 194K impressions, CTR 0.19%. **Wrong if:** this page recently changed URLs/titles and CTR hasn't caught up yet — a lag effect, not a real problem.
4. **content_acbcc847f8996314**: position 3.4, 171K impressions, CTR 0.15%. **Wrong if:** the ranking keyword doesn't match the page's actual content (intent mismatch): needs a content fix, not just title/snippet.
5. **content_471d9cabce329a66**: position 4.7, 165K impressions, CTR 0.24%. **Wrong if:** a competitor's brand or ad dominates that SERP, structurally capping CTR regardless of title quality.
6. **content_fd2117c2c6790e4b**: position 3.4, 151K impressions, CTR 0.27%. **Wrong if:** this is a navigational query where users already know the destination and skip straight to it via other means (bookmark, direct nav).
7. **content_34a70fea29d15f24**: position 3.2, 143K impressions, CTR 0.03%. **Wrong if:** this figure is a data artifact — CTR this low at position 3 is unusual enough to warrant checking for tracking/measurement issues before assuming it's a real title problem.
8. **content_e241d6415ac9e534**: position 3.3, 142K impressions, CTR 0.24%. **Wrong if:** seasonal demand dropped mid-March, deflating clicks temporarily — position stayed fine, but real-world interest shifted.
9. **content_f43118e089ecc69a**: position 5.0, 139K impressions, CTR 0.14%. **Wrong if:** this page cannibalizes traffic with a sibling page on the same site — clicks went to a related URL instead, not lost

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick:** #7 (content_34a70fea29d15f24) is the weakest, a CTR of 0.03% at position 3 is dramatically lower than every other top 10 pick, which is unusual enough to suspect a data or measurement anomaly rather than a genuine title/snippet problem. This should be manually inspected before acting on it.

**Leakage check:** No future-window data was used, all features (avg_position, impressions_total, ctr) come from the same March 2026 feature window as the decision point. No FlyRank product flags (health_score, priority_score, action_type) were used, since they aren't shipped in this dataset. No label derived inputs, since this is a rule, not a trained model there's no label to leak from.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.